# 4장 — 눈금 사이의 오차: 반올림과 누적 🌫️

웹 챕터의 세 사건(결합법칙 사망 · 티끌 경주 · 뺄셈 재앙)을 코드로 재현하고,
티끌 누적의 √n 성장(3부 7장의 랜덤워크!)을 **실측**합니다. 마지막엔 티끌 저금통(Kahan)도
직접 만들어요.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: 티끌의 행동 법칙.
**"순서가 반올림 지점을 바꾸고, 부호 제멋대로인 티끌은 √n 으로 쌓인다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch4')

## 1. 사건 ① · 결합법칙 사망: 같은 세 수, 다른 순서, 다른 답

In [ ]:
import numpy as np
f32 = np.float32

A = f32(f32(1) + f32(1e8)) - f32(1e8)   # (1 + 10⁸) − 10⁸ : 중간 결과에서 반올림!
B = f32(1) + (f32(1e8) - f32(1e8))       # 1 + (10⁸ − 10⁸) : 무사통과
print(f'(1 + 1e8) - 1e8 = {A}')
print(f'1 + (1e8 - 1e8) = {B}')

assert A == 0.0, '경로 A: 1이 10⁸ 옆에서 흡수(그 구간 눈금 간격 8) → 0'
assert B == 1.0, '경로 B: 0이 먼저 계산돼 1이 무사 → 1'
assert np.spacing(f32(1e8)) == 8.0, '범인 확인 — 10⁸ 구간의 fp32 눈금 간격은 8'
print()
print('✔ 확인: 계산은 맞고 반올림 지점이 다르다. fp 세계에 결합법칙은 없다')

## 2. 사건 ② · 티끌 경주: 순서가 티끌의 생사를 가른다

1.0 에 티끌(10⁻⁸)을 10만 번 더합니다. 두 가지 순서로.

In [ ]:
N, TINY = 100_000, f32(1e-8)

fwd = f32(1.0)
for _ in range(N):                 # 🐘 큰 수부터: 매번 흡수
    fwd = f32(fwd + TINY)

tiny_sum = f32(0.0)
for _ in range(N):                 # 🐜 작은 것끼리 먼저: 몸집을 키운다
    tiny_sum = f32(tiny_sum + TINY)
bwd = f32(1.0) + tiny_sum

print(f'큰 수부터   = {fwd}   (티끌 {N:,}개 전멸)')
print(f'작은 것 먼저 = {bwd}   (모은 티끌 {tiny_sum:.3e} 이 살아서 합류)')
print(f'참값        = {1 + N * 1e-8}')

assert fwd == 1.0, '흡수 — 티끌이 눈금 간격 절반(≈6×10⁻⁸)보다 작아 매번 제자리로 반올림'
assert abs(float(bwd) - 1.001) < 1e-6, '작은 것끼리 먼저 모으면 살아남는다'
print()
print('✔ 확인: 작은 것끼리 먼저 더해 몸집을 키운 뒤 큰 수와 합쳐라 (실전 요령)')

## 3. 사건 ③ · 뺄셈 재앙: 상대 오차의 폭발

(1 + d) − 1 의 참값은 d. 두 수가 가까워질수록(d ↓) 답의 **상대 오차**가 어떻게 되는지.

In [ ]:
print('    d        fp32 의 (1+d)−1     상대 오차')
rels = {}
for k in [4, 5, 6, 7, 8]:
    d = 10.0 ** -k
    diff = float(f32(f32(1 + d) - f32(1)))
    rel = abs(diff - d) / d * 100
    rels[k] = rel
    print(f'  1e-{k}     {diff:14.6e}     {rel:8.4g}%')

assert rels[4] < 0.1, '멀리 있는 두 수의 차는 안전(상대 오차 0.1% 미만)'
assert 10 < rels[7] < 30, 'd=10⁻⁷: 앞자리가 지워지며 티끌이 몸통으로 — 상대 오차 약 19%'
assert rels[8] == 100.0, 'd=10⁻⁸: 흡수로 정보 전멸 — 상대 오차 100%'
print()
print('✔ 확인: 뺄셈 자체는 정확하다. 몸통이 상쇄된 뒤 남는 것이 입력의 티끌뿐인 게 재앙')

## 4. 티끌 누적 실측: 상대 오차의 √n 성장 (3부 7장 랜덤워크)

무작위 값 n개를 fp32 로 순서대로 더하며(시드 고정) **값 대비 오차(상대 오차)**를 잽니다.
n이 10배가 될 때 상대 오차는 몇 배가 될까요? (n배가 아니라는 게 핵심!)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(9)                 # 시드 고정 — 결정적
data = rng.random(1_000_000).astype(np.float32)

ns, abs_errs, rel_errs = [10**3, 10**4, 10**5, 10**6], [], []
for n in ns:
    s = f32(0.0)
    for x in data[:n]:                          # 순진한 순차 합산 (fp32)
        s = f32(s + x)
    exact = float(np.sum(data[:n], dtype=np.float64))
    abs_errs.append(abs(float(s) - exact))
    rel_errs.append(abs_errs[-1] / exact)
    print(f'n = {n:>9,}   상대 오차 = {rel_errs[-1]:.3e}  (= ε 의 약 {rel_errs[-1] / 2**-23:,.0f}배)')

r1, r2 = rel_errs[2] / rel_errs[1], rel_errs[3] / rel_errs[2]   # n 10배당 상대 오차 비율
print(f'\nn 이 10배일 때 상대 오차 비율: {r1:.1f}배, {r2:.1f}배  (√10 ≈ 3.2 근처, 10배가 아니다!)')
assert rel_errs[3] > rel_errs[1], '상대 오차는 자란다'
assert r1 < 8 and r2 < 8, 'n에 비례(10배)하지 않고 √n 급으로 — 랜덤워크(3부 7장)'
assert rel_errs[3] < 1e-3, '100만 번을 더해도 상대 오차는 0.1% 미만 — fp 계산이 실용인 이유'

plt.figure(figsize=(6.5, 3.8))
plt.loglog(ns, rel_errs, 'o-', color='tab:purple', label='실측 상대 오차')
ref = [rel_errs[0] * (n / ns[0]) ** 0.5 for n in ns]
plt.loglog(ns, ref, '--', color='tab:gray', label='√n 기준선')
plt.xlabel('더한 개수 n'); plt.ylabel('상대 오차 |fp32 합 − 정확| / 정확'); plt.legend(); plt.grid(alpha=0.3, which='both')
plt.title('상대 오차는 n이 아니라 √n 으로: 시리즈 다섯 번째 √n'); plt.tight_layout(); plt.show()
print('✔ 확인: 부호 제멋대로인 티끌의 누적 = 랜덤워크. fp 계산이 실용인 이유')

## 5. 티끌 저금통: Kahan 합산 5줄

잃어버린 티끌을 장부(c)에 적어 뒀다가 다음 덧셈에 되돌려줍니다.

In [ ]:
def kahan_sum_f32(xs):
    """티끌 저금통 — 매 덧셈에서 잃은 티끌을 c 에 보관해 다음 항에 얹는다."""
    s, c = f32(0.0), f32(0.0)
    for x in xs:
        y = f32(x - c)              # 이번 항 + 저금통 잔액
        t = f32(s + y)              # 더하면서 티끌을 잃는다
        c = f32(f32(t - s) - y)     # 방금 잃은 티끌을 계산해 저금
        s = t
    return float(s)

n = 1_000_000
naive_err = abs_errs[-1]
kahan_err = abs(kahan_sum_f32(data[:n]) - float(np.sum(data[:n], dtype=np.float64)))
print(f'순진한 합산 오차 = {naive_err:.4g}')
print(f'Kahan  합산 오차 = {kahan_err:.4g}   ({naive_err / max(kahan_err, 1e-12):,.0f}배 개선)')

assert kahan_err < naive_err / 100, '5줄짜리 저금통으로 오차 수백 배 이상 개선'
print()
print('✔ 확인: 라이브러리 제작자들은 이 장의 사건을 이미 알고 설계에 반영해 뒀다 (NumPy 는 pairwise)')

**직접 해보기**: ① 4절의 시드(9)를 바꿔도 √n 경향은 그대로인지 확인해 보세요.
② `np.sum(data, dtype=np.float32)` 의 오차를 재 보세요. 순진한 루프보다 훨씬 작습니다
(pairwise 합산 덕분). GPU 가 덩어리로 나눠 더하는 것도 같은 계열인데, **나누는 방식이
실행마다 달라질 수 있다**는 게 재현성 이슈의 뿌리입니다.

## 6. 관찰 정리 + 🤖 AI 연결

- **머신 엡실론**(fp32: 2⁻²³ ≈ 1.19×10⁻⁷) = 1 근처 눈금 간격. 절반보다 작은 변화는 제자리로.
- 사건 3종: **결합법칙 사망**(순서가 반올림 지점을 바꾼다) · **흡수**(큰 수 옆 작은 수 전멸.
  대책: 작은 것끼리 먼저) · **유효숫자 손실**(가까운 두 수의 차, 상대 오차 폭발).
- 티끌 누적은 **√n**(랜덤워크, 3부 7장). 100만 연산에 천 배 수준. Kahan 저금통으로 수백 배 개선.
- **AI 연결**: GPU 병렬 합산은 나누는 순서가 실행마다 달라질 수 있다 → 로그확률 티끌 →
  temperature 0 에서도 박빙 토큰 순위가 뒤집힐 수 있다("같은 모델, 다른 답"의 수치적 범인).
  완전 재현이 필요하면 결정적 모드(비용: 속도).

**다음 챕터**: 5장(자의 끝). 너무 커서 터지는 inf, 너무 작아 사라지는 언더플로,
그리고 계산 불능의 표식 NaN. 8부 9장에서 본 손실 스파이크의 하드웨어적 얼굴.